# Compare retrieval baselines

Run the BM25, dense, and RRF retrieval baselines on the same development data, then compare summary metrics.

In [ ]:
%cd ..

import pandas as pd

from src.bm25 import Bm25Config
from src.dense_st import BgeM3Config, GteBaseConfig, Qwen3Embedding06BConfig
from src.io_utils import read_json
from src.paths import SYNTHETIC_DATA_DIR
from src.pipeline import ExperimentConfig, run_experiment
from src.rrf_backend import RRFConfig

query_path = SYNTHETIC_DATA_DIR / "gemma4" / "queries.jsonl"
qrels_path = SYNTHETIC_DATA_DIR / "gemma4" / "qrels.txt"

## BM25

In [ ]:
bm25_config = ExperimentConfig(
    experiment_name="bm25-full-document-baseline",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=Bm25Config(),
)

bm25_config

In [ ]:
bm25_experiment_dir = run_experiment(bm25_config)
bm25_experiment = read_json(bm25_experiment_dir / "experiment.json")

bm25_artifact_paths = {
    "experiment_dir": bm25_experiment_dir,
    "index_dir": bm25_experiment["index_dir"],
    "run_path": bm25_experiment_dir / "run" / "run.trec",
    "summary_path": bm25_experiment_dir / "eval" / "summary.json",
    "per_query_path": bm25_experiment_dir / "eval" / "per_query_metrics.jsonl",
}

bm25_artifact_paths

In [ ]:
bm25_summary = read_json(bm25_experiment_dir / "eval" / "summary.json")
bm25_summary_df = pd.DataFrame([bm25_summary]).T.rename(columns={0: bm25_config.backend.model_name})
bm25_summary_df


## GTE-Base

In [ ]:
gte_config = ExperimentConfig(
    experiment_name="gte-base-structure-aware-chunking-baseline",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=GteBaseConfig(),
)

gte_config


In [ ]:
gte_experiment_dir = run_experiment(gte_config)
gte_experiment = read_json(gte_experiment_dir / "experiment.json")

gte_artifact_paths = {
    "experiment_dir": gte_experiment_dir,
    "index_dir": gte_experiment["index_dir"],
    "run_path": gte_experiment_dir / "run" / "run.trec",
    "summary_path": gte_experiment_dir / "eval" / "summary.json",
    "per_query_path": gte_experiment_dir / "eval" / "per_query_metrics.jsonl",
}

gte_artifact_paths

In [ ]:
gte_summary = read_json(gte_experiment_dir / "eval" / "summary.json")
gte_summary_df = pd.DataFrame([gte_summary]).T.rename(columns={0: gte_config.backend.model_name})
gte_summary_df


## BGE-M3

In [ ]:
bge_m3_config = ExperimentConfig(
    experiment_name="bge-m3-structure-aware-chunking-baseline",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=BgeM3Config(),
)

bge_m3_config


In [ ]:
bge_m3_experiment_dir = run_experiment(bge_m3_config)
bge_m3_experiment = read_json(bge_m3_experiment_dir / "experiment.json")

bge_m3_artifact_paths = {
    "experiment_dir": bge_m3_experiment_dir,
    "index_dir": bge_m3_experiment["index_dir"],
    "run_path": bge_m3_experiment_dir / "run" / "run.trec",
    "summary_path": bge_m3_experiment_dir / "eval" / "summary.json",
    "per_query_path": bge_m3_experiment_dir / "eval" / "per_query_metrics.jsonl",
}

bge_m3_artifact_paths

In [ ]:
bge_m3_summary = read_json(bge_m3_experiment_dir / "eval" / "summary.json")
bge_m3_summary_df = pd.DataFrame([bge_m3_summary]).T.rename(columns={0: bge_m3_config.backend.model_name})
bge_m3_summary_df


## Qwen3-Embedding-0.6B

In [ ]:
qwen3_config = ExperimentConfig(
    experiment_name="qwen3-embedding-0.6b-structure-aware-chunking-baseline",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=Qwen3Embedding06BConfig(),
)

qwen3_config


In [ ]:
qwen3_experiment_dir = run_experiment(qwen3_config)
qwen3_experiment = read_json(qwen3_experiment_dir / "experiment.json")

qwen3_artifact_paths = {
    "experiment_dir": qwen3_experiment_dir,
    "index_dir": qwen3_experiment["index_dir"],
    "run_path": qwen3_experiment_dir / "run" / "run.trec",
    "summary_path": qwen3_experiment_dir / "eval" / "summary.json",
    "per_query_path": qwen3_experiment_dir / "eval" / "per_query_metrics.jsonl",
}

qwen3_artifact_paths

In [ ]:
qwen3_summary = read_json(qwen3_experiment_dir / "eval" / "summary.json")
qwen3_summary_df = pd.DataFrame([qwen3_summary]).T.rename(columns={0: qwen3_config.backend.model_name})
qwen3_summary_df


## RRF

In [ ]:
rrf_config = ExperimentConfig(
    experiment_name="rrf-bm25-gte-base-structure-aware-chunking-baseline",
    query_path=query_path,
    qrels_path=qrels_path,
    backend=RRFConfig(children=(Bm25Config(), GteBaseConfig())),
)

rrf_config


In [ ]:
rrf_experiment_dir = run_experiment(rrf_config)
rrf_experiment = read_json(rrf_experiment_dir / "experiment.json")

rrf_artifact_paths = {
    "experiment_dir": rrf_experiment_dir,
    "index_dir": rrf_experiment["index_dir"],
    "run_path": rrf_experiment_dir / "run" / "run.trec",
    "summary_path": rrf_experiment_dir / "eval" / "summary.json",
    "per_query_path": rrf_experiment_dir / "eval" / "per_query_metrics.jsonl",
}

rrf_artifact_paths

In [ ]:
rrf_summary = read_json(rrf_experiment_dir / "eval" / "summary.json")
rrf_summary_df = pd.DataFrame([rrf_summary]).T.rename(columns={0: rrf_config.backend.model_name})
rrf_summary_df


## Compare metrics

In [ ]:
comparison_df = pd.DataFrame(
    {
        bm25_config.backend.model_name: bm25_summary,
        bge_m3_config.backend.model_name: bge_m3_summary,
        gte_config.backend.model_name: gte_summary,
        qwen3_config.backend.model_name: qwen3_summary,
        rrf_config.backend.model_name: rrf_summary,
    }
)
comparison_df
